# Deep Neural Network
Trained on the entire dataset to identify between tumor and normal samples

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import keras
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## 1. Load, clean & map gene names

In [32]:
#reading in csv file
df = pd.read_csv('denseDataOnlyDownload-2.tsv', sep='\t')

#mapping csv file names to gene markers, thenrenaming columns
ensembl_map = {
    'ENSG00000082175.15': 'PGR',
    'ENSG00000141736.14': 'ERBB2',
    'ENSG00000087586.18': 'SLAMF7',
    'ENSG00000171862.11': 'PTEN',
    'ENSG00000146648.19': 'EGFR',
    'ENSG00000103855.18': 'CD276',
    'ENSG00000090339.9':  'ICAM1'
}
df = df.rename(columns=ensembl_map)
df = df.rename(columns={'sample_type.samples':'sample_type','primary_diagnosis.diagnoses':'primary_diagnosis'})
GENES = list(ensembl_map.values())

# Remove nulls
before = len(df)
df = df.dropna(subset=GENES + ['sample_type'])
print(f'Removed {before-len(df)} rows with nulls → {len(df)} samples remaining')
print(f'Sample types:\n{df["sample_type"].value_counts()}')

Removed 31 rows with nulls → 1226 samples remaining
Sample types:
sample_type
Primary Tumor          1106
Solid Tissue Normal     113
Metastatic                7
Name: count, dtype: int64


## 2. Split tumor vs normal

In [34]:
#combining Primary Tumor and Metastatics into one type
# combining dataframes into one
tumor  = df[df['sample_type'].isin(['Primary Tumor','Metastatic'])].copy()
normal = df[df['sample_type'] == 'Solid Tissue Normal'].copy()
combined = pd.concat([tumor, normal], ignore_index=True)
combined['label'] = (combined['sample_type'].isin(['Primary Tumor','Metastatic'])).astype(int)
print(f'Tumor samples:  {len(tumor)}')
print(f'Normal samples: {len(normal)}')

Tumor samples:  1113
Normal samples: 113


In [35]:
#showing combined dataframe head
print(combined.shape)
combined.head()

(1226, 15)


,sample,samples,sample_type,disease_type,primary_diagnosis,primary_site,disease_type.1,PGR,ERBB2,SLAMF7,PTEN,EGFR,CD276,ICAM1,label
0,TCGA-AC-A6IX-06A,TCGA-AC-A6IX-06A,Metastatic,Ductal and Lobular Neoplasms,"Lobular carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,0.2133,4.851,2.016,3.410,1.5460,4.185,3.049,1
1,TCGA-E2-A15E-06A,TCGA-E2-A15E-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,5.9380,7.051,2.208,4.504,0.6054,4.580,2.958,1
2,TCGA-E2-A15A-06A,TCGA-E2-A15A-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,2.8910,5.271,4.435,2.536,0.9872,4.173,3.025,1
3,TCGA-BH-A1FE-06A,TCGA-BH-A1FE-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,2.8380,3.028,2.612,3.743,3.2020,5.287,5.570,1
4,TCGA-BH-A1ES-06A,TCGA-BH-A1ES-06A,Metastatic,Ductal and Lobular Neoplasms,"Infiltrating duct carcinoma, NOS",Breast,Ductal and Lobular Neoplasms,2.5050,5.889,2.560,3.898,1.6520,4.293,4.300,1


## 3. Defining Deep Neural Network
Defining from overall structure from input to output. 

In [37]:
#setting model input to number of columns/features in samples
inputs = keras.Input(shape=(7,))

#defining model's hidden layers
x = keras.layers.Dense(256,activation='relu')(inputs)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.25)(x)  

x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.25)(x)

x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.25)(x)

x = keras.layers.Dense(32, activation='relu')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.25)(x)

#defining output layer 
outputs = keras.layers.Dense(1, activation='softmax')(x)


## 4. Training and Testing Model
Using Stratified Kfold to ensure each fold maintains the original class distribution. Performing SMOTE on each fold to generate synthethic minority class samples to balance training data.

In [39]:
# setting up feature and target variables
features = ['PGR', 'ERBB2', 'SLAMF7', 'PTEN', 'EGFR', 'CD276', 'ICAM1']
X = combined[features]
y = combined['label']

#setting up K-folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#array to store each fold results
fold_metrics = []

#iterating through folds
for i, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    #print progress
    print(f"Training Fold {i+1}...")

    # Use .iloc to safely index pandas objects by position
    X_train, X_test = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[val_idx]

    # Initialize SMOTE
    sm = SMOTE(random_state=42)

    # Resample the training data
    X_res, y_res = sm.fit_resample(X_train, y_train)

    #creating and compiling new model each fold
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy',keras.metrics.F1Score(name='f1_score')])

    #traing using fold indices
    model.fit(X_res, y_res, epochs=50, verbose=0)

    # evaluating on fold indices
    loss, acc, f1 = model.evaluate(X_test, y_test, verbose=0)
    fold_metrics.append({'accuracy':acc,'f1_score':f1})

Training Fold 1...
Training Fold 2...
Training Fold 3...
Training Fold 4...
Training Fold 5...


## 5. Evaluating model
Averaging each K-fold metric and returning results.

In [41]:
#printing average metrics
acc_mean = sum(d['accuracy'] for d in fold_metrics) / 5.0
f1_mean = sum(d['f1_score'] for d in fold_metrics) / 5.0
print(f"Accuracy: {acc_mean:.4f}")
print(f"F1_score:  {f1_mean: .4f}")

Accuracy: 0.9078
F1_score:   0.9517
